In [1]:
!pip install bertopic sentence-transformers umap-learn hdbscan

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 81.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 91.5 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 92.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 94.8 MB/s  0:00:00
  Created wheel for hdbscan: filename=hdbscan-0.8.41-cp313-cp313-linux_x86_64.whl size=4396448 sha256=f76638819ef75120ac653a3d78f8fbc897e9d2f816f84b1e5a4b6455dbeff726
  Stored in directory: /home/onyxia/.cache/pip/wheels/c8/bc/55/6d2928daed92cff0ff2b4c592c9c09546c3168ba0faeb610df
Successfully built hdbscan
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [bertopic]7/9 [sentence-transformers]


In [ ]:
# Si hdbscan marche pas on peut utiliser ça

from sklearn.cluster import HDBSCAN
from bertopic.cluster import BaseCluster
"""
class SklearnHDBSCAN(BaseCluster):
    def __init__(self, *args, **kwargs):
        self.model = HDBSCAN(*args, **kwargs)
    
    def fit(self, X):
        self.model.fit(X)
        self.labels_ = self.model.labels_
        return self
    
    def fit_predict(self, X):
        self.labels_ = self.model.fit_predict(X)
        return self.labels_
hdbscan_model = SklearnHDBSCAN(
    min_cluster_size=50,      # bigger = fewer, larger topics
    min_samples=10,            # higher = more conservative clustering
    cluster_selection_epsilon=0.1,  # merge nearby clusters
    cluster_selection_method='leaf' # fewer, more specific clusters
)
"""

Récupération des fichiers CSV

In [2]:
import pandas as pd
import os
import s3fs



fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

s3_folder = "biath"

files = fs.glob(f"s3://lab/PESSD/csv/{s3_folder}/*.csv")

for file in files:
    fs.get(file, "data/"+(file.removeprefix(f"lab/PESSD/csv/{s3_folder}")))

In [25]:
df = pd.DataFrame()
for file in os.listdir("data") : 
    temp = pd.read_csv("data/"+file)
    df = pd.concat([df,temp])


Nettoyage de la data

In [27]:
import re
import string

# Removing music and noise (only technical commentary)
df = df[(df["main_g"]=="female")|(df["main_g"]=="male")]

# text to sentences
df["text"] = df["text"].apply(lambda x : x.split(". "))
df = df.explode("text")

# Removes all words with capitals
df["text"] = df["text"].apply(lambda x : re.sub(r"\s*[A-Z]\w*\s*", " ", x).strip())

# Remove punctuation
df ["text"] = df["text"].apply(lambda x : x.translate(str.maketrans('', '', string.punctuation.replace("'", "").replace("-", ""))))



On ajoute les métadonnées

In [28]:
df["ID"] = df["audio_file"].str.replace("_wav.wav","")
df = df.merge(pd.read_csv("metadata.csv"), on="ID")

In [29]:
df

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath
0,0.00,12.20,on va prier en ce dimanche pour l'équipe de a...,SPEAKER_02,male,BAH_wav.wav,BAH,H
1,12.20,28.80,ça,SPEAKER_03,male,BAH_wav.wav,BAH,H
2,12.20,28.80,au moment où lance une attaque avec son coéq...,SPEAKER_03,male,BAH_wav.wav,BAH,H
3,28.80,31.80,surtout dans cette zone oui,SPEAKER_08,male,BAH_wav.wav,BAH,H
4,31.80,38.80,travail des techniciens on voit la glisse des,SPEAKER_02,male,BAH_wav.wav,BAH,H
...,...,...,...,...,...,...,...,...
8842,2893.44,2919.36,il y a - qui est là,SPEAKER_02,male,BMH_wav.wav,BMH,H
8843,2893.44,2919.36,- on va garder ce chiffre en tête en tête à r...,SPEAKER_02,male,BMH_wav.wav,BMH,H
8844,2893.44,2919.36,devient de toute l'histoire des le français l...,SPEAKER_02,male,BMH_wav.wav,BMH,H
8845,2893.44,2919.36,sacré cours et sacré jeu pour,SPEAKER_02,male,BMH_wav.wav,BMH,H


In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
import hdbscan

# Stopwords list
STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

# Docs 
docs = df["text"]

# Vectorizer for topic representation
vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1,2),
    min_df=2,
    token_pattern=r"\b(?!\w*[A-Z])\w+\b" # theoreticaly removes words with capital letters once embeddings are done
)

# To control for number and size of clusters
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=50,      # bigger = fewer, larger topics
    min_samples=10,            # higher = more conservative clustering
    cluster_selection_epsilon=0.1,  # merge nearby clusters
    cluster_selection_method='leaf', # fewer, more specific clusters
    prediction_data=True # important !!
)

# Load CamemBERT embedding model
embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device="cuda") # remove device if CPU

# Influence choice of themes
seed =  [
["émotion","larme","ému","joie","tristesse"],
["famille","parents","ami", "personnel", "conjoint"]]

# Build BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=seed    
)

# Fit the model
topics, probs = topic_model.fit_transform(docs)

# Show discovered topics
print(topic_model.get_topic_info())

# Visualize topics
#topic_model.visualize_topics()

# Visualize topic hierarchy
topic_model.visualize_hierarchy()

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1774620027.085034   15956 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774620027.147835   15956 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774620028.451870   15956 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point rou

On regroupe les phrases par topic

In [9]:
tp = pd.DataFrame({
    "text": docs,
    "topic": topics,
    "topic_prob": probs.max(axis=1)  # max probability per text
})

In [14]:
df = df.merge(tp, on="text")
df

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath,topic,topic_prob
0,7.08,17.88,les ...,SPEAKER_02,female,ELC_wav.wav,ELC,M,-1,0.059391
1,7.08,17.88,Les mamansament font lui,SPEAKER_02,female,ELC_wav.wav,ELC,M,0,1.000000
2,18.04,4.98,des,SPEAKER_00,female,ELC_wav.wav,ELC,M,-1,0.052688
3,19.16,52.68,iniciants,SPEAKER_02,female,ELC_wav.wav,ELC,M,9,1.000000
4,19.16,52.68,Ils n'entraînent pas et n'en peuvent pas,SPEAKER_02,female,ELC_wav.wav,ELC,M,15,1.000000
...,...,...,...,...,...,...,...,...,...,...
15025,1916.16,1949.48,Et passer le cut,SPEAKER_04,female,DR2_wav.wav,DR2,F,-1,0.024190
15026,1916.16,1949.48,"Merci Nathalie et Péchala, merci Annick Dumont",SPEAKER_04,female,DR2_wav.wav,DR2,F,19,1.000000
15027,1916.16,1949.48,"Merci Marie-Christine, Marie",SPEAKER_04,female,DR2_wav.wav,DR2,F,19,0.383342
15028,1916.16,1949.48,On se retrouve demain pour de nouvelles aventures,SPEAKER_04,female,DR2_wav.wav,DR2,F,-1,0.086515


In [23]:
# Sentences of topic i

for k in df[df["topic"]==1]["text"] : 
    print(k)

C'est ça qui est hyper impressionnant
Tout était incroyable et fantastique.
Jamais de la vie
Il est super
La sensation, elle doit être incroyable
Et brillant
C'était un truc pour...
Triple bout, parfait.
Et c'est très bien
C'est très très rare ce que l'on vient de voir
Ça aussi, c'est incroyable
C'est très rare
Mais c'est quand même exceptionnel.
Ils arrivent vraiment à nous embarquer dans leur histoire.
J'adore cette image arrêtée, c'est beau
Je l'adore, c'est super.
Le score, incroyable
Même là, c'est limite.
Très souvent, tout cas Daniel.
Parfois ça ne passe pas, mais quand ça passe, qu'est-ce que c'est bon
C'est extra
Ça c'est super
No, I wish the sea wonder
C'est extraordinaire
C'est extraordinaire
Moi, j'ai adoré
C'était une pure merveille
Tout est subtil
C'est absolument divin
J'ai adoré
Donc ça serait extraordinaire
Sur le reste c'est magnifique
Mais elle va être excellente
Alors moi je l'adore à elle
portée là, qui est magnifique
Niveau 4 oui, mais surtout c'est original, c'es

Label count per topic

In [31]:
# Only classified docs
clf = df[df["topic"]!=-1]
cdf = pd.DataFrame()
for i in range(len(clf["topic"].unique())) :
    tmp = df[df["topic"]==i]
    tmp = tmp["g_ath"].value_counts(normalize=True)
    tmp = pd.DataFrame([{"topic":f"topic_{i}","M":tmp["M"],"F":tmp["F"],"H":tmp["H"]}])
    cdf = pd.concat([cdf,tmp])

# All topics
tmp = clf["g_ath"].value_counts(normalize=True)
tmp = pd.DataFrame([{"topic":f"all_topics","M":tmp["M"],"F":tmp["F"],"H":tmp["H"]}])
cdf = pd.concat([cdf,tmp])

cdf


,topic,M,F,H
0,topic_0,0.806557,0.118033,0.075410
0,topic_1,0.423358,0.317518,0.259124
0,topic_2,0.588028,0.207746,0.204225
0,topic_3,0.500000,0.253846,0.246154
0,topic_4,0.553435,0.270992,0.175573
0,topic_5,0.547264,0.313433,0.139303
0,topic_6,0.450292,0.374269,0.175439
0,topic_7,0.670968,0.206452,0.122581
0,topic_8,0.552448,0.265734,0.181818
0,topic_9,0.534483,0.310345,0.155172


Tests de stabilité (GPU nécessaire)

In [15]:
# Main words of current model themes

reference = set()
for topic_id in topic_model.get_topics():
    if topic_id == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(topic_id)[:10]]
    reference.add(tuple(sorted(words)))
reference

{('1',
  '8',
  'attention',
  'bonsoir',
  'fantastique',
  'ok',
  'oui',
  'oui oui',
  'oui thème',
  'thème'),
 ('102', '102 55', '46', '54', '55', '62', '64', '70', '92', 'mesure'),
 ('2022',
  '2023',
  '2024',
  '2025',
  'champion',
  'champion monde',
  'europe',
  'monde',
  'monde 2023',
  'médailles'),
 ('23',
  'applaudissements',
  'attention',
  'bascule',
  'limite',
  'maxime',
  'maximus',
  'parfait',
  'suspense',
  'également'),
 ('2ème',
  '2ème note',
  'artistique',
  'monte',
  'monter',
  'note',
  'note artistique',
  'note note',
  'notes',
  'scorez'),
 ('4 tours',
  'axel tours',
  'demi',
  'seconde',
  'tour',
  'tour tours',
  'tours',
  'tours demi',
  'tours tour',
  'tours tours'),
 ('5',
  '5 points',
  'bonus',
  'bonus malus',
  'donner',
  'juges',
  'jury',
  'malus',
  'points',
  'élément'),
 ('71',
  '72',
  '78',
  '89',
  'meilleur',
  'meilleur score',
  'record',
  'saison',
  'score',
  'score saison'),
 ('accord',
  'complètement',
  '

On run 10 fois le même modèle à 80% du corpus pour voir la stabilité des thèmes

In [16]:
from sklearn.utils import resample
from collections import Counter

# List of words for each topic for each model
all_topics_words = []

# Number of runs
nb_iter = 10

for i in range(nb_iter):

    # First, resample 80% of docs
    sample = resample(docs, n_samples=int(len(docs) * 0.8), random_state=i)

    # Same model specification
    model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=seed    
)
    model.fit(sample)
    
    # Top 10 words per topic
    top_words = set()
    for topic_id in model.get_topics():
        if topic_id == -1:
            continue
        words = [w for w, _ in model.get_topic(topic_id)[:10]]
        top_words.add(tuple(sorted(words)))
    
    all_topics_words.append(top_words)


# Jaccard similarity index
def jaccard(t1, t2):
    s1, s2 = set(t1), set(t2)
    return len(s1 & s2) / len(s1 | s2)

# Topic stability
"""
Measures similarity between reference topic and topics found in other runs.

Thresholds for stability are determined by Jaccard index (threshold) and number of occurences (min_run)
"""
def is_stable(topic, all_runs, threshold=0.2, min_runs=7):
    count = 0
    for run_topics in all_runs:
        if any(jaccard(topic, t) >= threshold for t in run_topics):
            count += 1
    return count >= min_runs


stable_topics = [t for t in reference if is_stable(t, all_topics_words)]

print(f"Thèmes stables (≥7 runs sur 10) : {len(stable_topics)}")
for t in stable_topics:
    print(t)


2026-03-22 16:11:13,532 - BERTopic - Embedding - Transforming documents to embeddings.


Batches: 100%|██████████| 197/197 [00:09<00:00, 21.72it/s]
2026-03-22 16:11:22,678 - BERTopic - Embedding - Completed ✓
2026-03-22 16:11:22,679 - BERTopic - Guided - Find embeddings highly related to seeded topics.
Batches: 100%|██████████| 1/1 [00:00<00:00, 85.72it/s]
2026-03-22 16:11:22,744 - BERTopic - Guided - Completed ✓
2026-03-22 16:11:22,747 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-22 16:12:00,537 - BERTopic - Dimensionality - Completed ✓
2026-03-22 16:12:00,539 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-22 16:12:01,352 - BERTopic - Cluster - Completed ✓
2026-03-22 16:12:01,361 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-22 16:12:01,520 - BERTopic - Representation - Completed ✓
2026-03-22 16:12:01,726 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 197/197 [00:09<00:00, 21.73it/s]
2026-03-22 16:12:10,860 - BERTopic - Embeddin

Thèmes stables (≥7 runs sur 10) : 21
('2ème', '2ème note', 'artistique', 'monte', 'monter', 'note', 'note artistique', 'note note', 'notes', 'scorez')
('air', 'axel', 'combinaison', 'niveau', 'petit', 'points', 'saut', 'sauts', 'technique', 'triple')
('copie danse', 'danse', 'danse glace', 'discipline', 'feu', 'glace', 'glace couple', 'milan', 'patinoire', 'référence')
('boléro', 'choix', 'chorégraphie', 'chorégraphique', 'danse', 'interprétation', 'interprété', 'musical', 'musique', 'rythme')
('beaudry', 'cizeron', 'fournier', 'geoffrey', 'guillaume', 'guillaume cizeron', 'guillaume laurence', 'laurence', 'laurence fournier', 'laurence guillaume')
('5', '5 points', 'bonus', 'bonus malus', 'donner', 'juges', 'jury', 'malus', 'points', 'élément')
('court', 'libre', 'libre programme', 'meilleur programme', 'programme', 'programme court', 'programme libre', 'programme programme', 'programmes', 'voir programme')
('maximum', 'maximum niveau', 'niveau', 'niveau 4', 'niveau maximum', 'parfait

TODO ajouter la stabilité des labels 

keybertopic, extrait phrases saillantes
tester lda
virer thèmes nuls et rerun
checker distance cosine